# Co-occurrence & Graphe de Synergies
**Objectif :** identifier quelles cartes apparaissent systématiquement ensemble dans les decks gagnants.

**Approche :**
1. Charger toutes les cartes du main deck par tournoi deck
2. Construire une matrice deck × carte (1 si présente, 0 sinon)
3. Calculer la co-occurrence : `matrice.T @ matrice`
4. Normaliser pour obtenir un score entre 0 et 1
5. Construire le graphe NetworkX

In [ ]:
import sqlite3
import pandas as pd
import numpy as np

con = sqlite3.connect('../data/yugioh.db')

# Charger uniquement les cartes du main deck (pas extra/side)
df = pd.read_sql("""
    SELECT dc.deck_id, dc.card_name, td.archetype, td.placement, td.uploaded
    FROM deck_cards dc
    JOIN tournament_decks td ON td.id = dc.deck_id
    WHERE dc.zone = 'main'
    AND td.illegal = 0
""", con)

print(f'Lignes chargées : {len(df):,}')
print(f'Decks uniques   : {df["deck_id"].nunique():,}')
print(f'Cartes uniques  : {df["card_name"].nunique():,}')

## 1. Matrice deck × carte

In [ ]:
# Matrice binaire : 1 si la carte est dans le deck (on ignore les quantités pour l'instant)
matrix = df.groupby(['deck_id', 'card_name']).size().unstack(fill_value=0).clip(upper=1)

print(f'Matrice : {matrix.shape[0]} decks × {matrix.shape[1]} cartes')

## 2. Calcul de co-occurrence

In [ ]:
# Co-occurrence brute : nombre de decks où les deux cartes apparaissent ensemble
m = matrix.values  # numpy array
cooc_raw = m.T @ m  # shape: (nb_cartes, nb_cartes)

# Normalisation : on divise par le nombre de decks qui contiennent chaque carte
# score(A, B) = decks(A et B) / decks(A ou B)  → Jaccard similarity
card_counts = np.diag(cooc_raw)  # nombre de decks par carte
union = card_counts[:, None] + card_counts[None, :] - cooc_raw
jaccard = np.where(union > 0, cooc_raw / union, 0)
np.fill_diagonal(jaccard, 0)  # supprimer la diagonale (carte avec elle-même)

cards = matrix.columns.tolist()
cooc_df = pd.DataFrame(jaccard, index=cards, columns=cards)

print('Matrice de co-occurrence calculée.')
print(f'Shape : {cooc_df.shape}')

## 3. Top paires de cartes les plus liées

In [ ]:
# Extraire les paires (triangle supérieur pour éviter les doublons)
upper = np.triu(jaccard, k=1)
pairs_idx = np.argwhere(upper > 0.1)  # seuil minimum : 10% de Jaccard

pairs = []
for i, j in pairs_idx:
    pairs.append({
        'card_a': cards[i],
        'card_b': cards[j],
        'jaccard': round(jaccard[i, j], 4),
        'cooc_count': int(cooc_raw[i, j])
    })

pairs_df = pd.DataFrame(pairs).sort_values('jaccard', ascending=False)
print(f'Paires avec Jaccard > 0.1 : {len(pairs_df):,}')
pairs_df.head(20)

## 4. Co-occurrence par archetype

In [ ]:
def top_pairs_for_archetype(archetype, min_jaccard=0.5, top_n=15):
    """Retourne les paires de cartes les plus liées pour un archetype donné."""
    decks = df[df['archetype'] == archetype]['deck_id'].unique()
    sub = matrix.loc[matrix.index.isin(decks)]
    sub = sub.loc[:, sub.sum() > 0]  # garder seulement les cartes présentes
    
    if sub.shape[0] < 5:
        print(f'Pas assez de decks pour {archetype} ({sub.shape[0]})')
        return
    
    m2 = sub.values
    c2 = m2.T @ m2
    cnt = np.diag(c2)
    union2 = cnt[:, None] + cnt[None, :] - c2
    jac2 = np.where(union2 > 0, c2 / union2, 0)
    np.fill_diagonal(jac2, 0)
    
    local_cards = sub.columns.tolist()
    upper2 = np.triu(jac2, k=1)
    idx2 = np.argwhere(upper2 >= min_jaccard)
    
    result = [{'card_a': local_cards[i], 'card_b': local_cards[j],
               'jaccard': round(jac2[i,j], 3), 'count': int(c2[i,j])}
              for i, j in idx2]
    
    return pd.DataFrame(result).sort_values('jaccard', ascending=False).head(top_n)

# Exemple sur le top archetype
print('=== Maliss ===')
top_pairs_for_archetype('Maliss')

In [ ]:
print('=== Tenpai Dragon ===')
top_pairs_for_archetype('Tenpai Dragon')

In [ ]:
print('=== Snake-Eye ===')
top_pairs_for_archetype('Snake-Eye')

## 5. Cartes "staples" — présentes dans tous les archetypes

In [ ]:
# Une staple = carte présente dans beaucoup de decks, tous archetypes confondus
card_freq = matrix.sum(axis=0) / len(matrix)
staples = card_freq[card_freq > 0.3].sort_values(ascending=False)

print(f'Cartes présentes dans >30% des decks ({len(staples)} cartes) :')
for card, freq in staples.items():
    print(f'  {freq:.0%}  {card}')

## 6. Sauvegarder les paires en base

In [ ]:
# Sauvegarder les paires significatives (Jaccard > 0.05) dans SQLite
significant = pairs_df[pairs_df['jaccard'] > 0.05].copy()

con2 = sqlite3.connect('../data/yugioh.db')
con2.execute("DROP TABLE IF EXISTS card_cooccurrence")
con2.execute("""
    CREATE TABLE card_cooccurrence (
        card_a      TEXT,
        card_b      TEXT,
        jaccard     REAL,
        cooc_count  INTEGER,
        PRIMARY KEY (card_a, card_b)
    )
""")
significant.to_sql('card_cooccurrence', con2, if_exists='append', index=False)
con2.commit()
con2.close()

print(f'✓ {len(significant):,} paires sauvegardées dans card_cooccurrence')